# Разбор A. Реакция и наблюдение

Если при подготовке пробы меняется химическая форма, спектр и хроматограмма уже относятся к другой смеси. Начнём с элементарного контроля стехиометрии, затем рассмотрим условный переход кислота→лактон B. Для линии A опубликованная циклизация ксантинов даёт контекст реакции, но её выход не является константой скорости.

Сначала вычислите формулы и массы кислоты/лактона B, не меняя исходные записи. Затем решите условную ODE k=.02 min^-1 на 0…120 min и сравните с аналитическим решением. Постройте обе концентрации и проверьте баланс в каждой точке.

In [ ]:
from pathlib import Path
import sys,json,math
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'programm/course.json').is_file())
sys.path.insert(0,str(ROOT/'practicuum/shared'))
from course_support import DATA,compound,esol,workspace,write_json,init_dossier
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
def publish_plot(fig,name):
    fig.savefig(OUT/name,dpi=120,bbox_inches='tight')
    display(fig)
    plt.close(fig)
OUT=workspace('M09-L01','case')


Сначала проверяем входы и единицы. Словари сохраняют названия полей вместе со значениями; список сохраняет соответствие записей. Выведите промежуточный результат перед дальнейшим анализом.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors,rdMolDescriptors
forms=[Chem.MolFromSmiles(compound(n)['smiles']) for n in ['atorvastatin','atorvastatin_lactone']]
print([rdMolDescriptors.CalcMolFormula(m) for m in forms])
assert abs(Descriptors.MolWt(forms[0])-Descriptors.MolWt(forms[1])-18.015)<1e-6


Теперь сравним модель с контрольным случаем. `assert` останавливает расчёт при нарушении контракта; это защита от ошибочного продолжения, а не оценка научного понимания.

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
time=np.linspace(0,120,121);k=.02;initial=1e-4
sol=solve_ivp(lambda t,y:[-k*y[0],k*y[0]],[0,120],[initial,0],t_eval=time,rtol=1e-9,atol=1e-13)
assert sol.success
analytic=initial*np.exp(-k*time)
assert np.max(abs(sol.y[0]-analytic))<1e-11 and np.allclose(sol.y.sum(axis=0),initial,atol=1e-12)
fig,ax=plt.subplots();ax.plot(time,sol.y[0],label='acid model');ax.plot(time,sol.y[1],label='lactone model')
ax.set(xlabel='Time / min',ylabel='Simulated concentration / mol L^-1');ax.legend();publish_plot(fig,'kinetics.png')


Сохраним именно полученный результат. Не переносите этот вывод на другую химическую форму, мишень или экспериментальные условия без отдельной проверки.

In [ ]:
result={'kind':'simulation','k_min_inverse':k,'maximum_solver_error':float(np.max(abs(sol.y[0]-analytic))),
        'actual_B_stability_measurement':'NOT_RUN'}


In [ ]:
write_json(OUT/'result.json',result)
from dossier_records import append_record
receipt=append_record(init_dossier(),'M09-L01','A',result,ROOT,inputs=['practicuum/shared/data/compounds.json'])
print('Versioned evidence:',receipt)


Совпадение ODE с аналитикой проверяет реализацию. Оно не измеряет стабильность B и не доказывает механизм. Для константы скорости нужен временной эксперимент, который станет задачей следующего урока.

Перед переходом ответьте: какие результаты получены вычислением, какие заданы условно, и какая независимая проверка ещё отсутствует?